# 08 · Reachable corridor, checked against Shapely

## What this checks

**Reachable corridor** draws the ground someone can reach along streets within a travel budget. It follows each street
as far as the budget allows. Then it adds a band either side, for the walk off the street at the end of a route. This
check covers release 0.3.0.

Since 14 October 2026 that last walk is charged against the budget. The band at any point is the smaller of:

- the limit on the final walk, here 25 m;
- the budget left at that point.

So the corridor narrows to nothing where the time runs out. The route also starts at the site itself, not at the
junction nearest to it.

Before that change, the corridor was a strip of the same width around every street reached. That included streets at
the very edge of the budget. Someone standing there has no time left to walk, yet the corridor still claimed 25 m of
ground around them. This check was rewritten with the widget. The old check rebuilt a strip of one width, and agreed
with a Fieldwork that drew one. Both have changed, and so have the data.

The corridor is the area a team treats as within reach. So it is worth checking.

## How it is checked

Fieldwork declares its rule. We rebuild the corridor from that rule with **Shapely**, Python's link to GEOS. GEOS is
the geometry engine much GIS software uses. We do not compare shapes piece by piece. Nothing here uses Fieldwork's
code.

**Our rebuild is finer than Fieldwork, on purpose.** It steps along each route 1 m at a time, with the exact budget
left at each step. Fieldwork cuts each route into pieces of 12.5 m, and rounds the band's width down into six steps.
So our rebuild is the rule as stated. Fieldwork is an approximation of it, and that approximation is what we measure.

The data are the five-minute corridor in Soho, from the John Snow worked example. It reaches 134 junctions, with a
25 m limit on the final walk. The data carry the shape of each street, because these streets curve and the band
follows the curve.

## What should happen

1. **The same streets, exactly.** The same number of streets entered, and the same number cut part-way. Both sides
   choose streets by the same stated arithmetic over the same numbers. So a difference would be a difference in the
   rule, not in the geometry engine.
2. **Fieldwork's area at most 100.5% of ours.** Fieldwork's approximation pulls both ways, but neither way by much. A
   corridor clearly larger than the rule would mean the narrowing does not follow the budget. The half per cent also
   allows for two geometry engines and two projections.
3. **Fieldwork's area at least 85% of ours.** A shortfall beyond that would mean the narrowing bites where the rule
   does not put it.
4. **The two shapes differ by at most 8% of the area.** Equal areas can still be different shapes. So we also measure
   the ground in one corridor but not the other. This limit is looser than the 2% the old check used. The two
   approximations differ along the whole edge of the corridor, not only in total.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_08_corridor import run

report = run(Path.cwd() / "fixtures" / "corridor-soho-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(f"reached junctions: {report['reachedJunctions']}")
print(f"segments entered: Fieldwork {report['fieldwork']['enteredSegments']}, "
      f"Shapely {report['reference']['enteredSegments']}")
print(f"entered part-way: Fieldwork {report['fieldwork']['partialSegments']}, "
      f"Shapely {report['reference']['partialSegments']}")
print(f"area: Fieldwork {report['fieldwork']['areaSquareMetres']:.0f} m2, "
      f"Shapely {report['reference']['areaSquareMetres']:.0f} m2")
print(f"area difference        {report['areaDifferenceFraction']*100:.3f}%")
print(f"symmetric difference   {report['symmetricDifferenceFraction']*100:.3f}%")

## The result on record

First recorded on 14 October 2026, against release 0.2.0. Rerun on 26 November 2026, against release 0.3.0:
**Agrees**.

Both sides entered the same 283 streets, and cut the same 38 of them part-way. Fieldwork's corridor covers 161,044 m².
Ours covers 160,785 m². They are 0.160% apart, and the ground in one but not the other is 0.163% of the area.

Running the cell above repeats the check on the same data.

## Where the difference comes from

The choice of streets agrees exactly. That is the part that carries the method. It is rebuilt from the written rule,
not copied.

What is left is the narrowing, and it is worth saying which way it goes. Fieldwork draws **slightly more** ground than
the rule, not less. Two effects pull against each other:

- Rounding each piece's width down takes ground away.
- Fieldwork draws the band around each piece separately. So each piece's far end gets a round cap at that piece's
  width, where the rule gives a smaller one. That adds ground.

On this corridor the second effect is larger. The excess is about a sixth of a per cent. The ground in one shape but
not the other is about the same size. So the difference is spread around the edge, not a piece of corridor present in
one and missing from the other.

So Fieldwork's approximation does not always err on the small side. That is recorded here, and in the widget itself,
not assumed away.

## What this does not show

- That the streets exist and can be walked. Both sides trust the same street shapes and lengths. A wrong network is
  wrong in both, and this check still passes.
- That a 25 m final walk is right. No reason for it is recorded.
- That the ground beside a street can be walked at all. The corridor still counts the insides of buildings, locked
  yards and the far side of walls as reached.
- That Fieldwork's approximation is close enough for a particular use. It shows how close it is on this corridor.
- Any corridor but this one: the five-minute band of one corridor, outbound.